In [1]:
import pandas as pd
import numpy as np
import altair as alt
from vega_datasets import data


pd.__version__

'2.2.2'

Load in data

In [4]:
# 1. Load some sample data
df_CO2 = pd.read_excel("EDGAR_2025_GHG_booklet_2025.xlsx" , sheet_name='GHG_per_capita_by_country', nrows=210)


In [5]:
#Data Processing
df_CO2.rename(columns={'EDGAR Country Code': 'CountryCode'}, inplace=True)

print(df_CO2.columns.tolist())
# Reshape from wide to long
# Get all columns that are NOT 'CountryCode' or 'Country' (these are the year columns)
year_columns = [col for col in df_CO2.columns if col not in ['CountryCode', 'Country']]

# First, convert the year columns to numeric
for col in year_columns:
    df_CO2[col] = pd.to_numeric(df_CO2[col], errors='coerce')

# Melt
df_long = df_CO2.melt(id_vars=[df_CO2.columns[0], df_CO2.columns[1]],
                       var_name='Year',
                       value_name='GHG Emissions Per Capita')

#year to metric
df_long['Year'] = pd.to_numeric(df_long['Year'])

#NaN emissions
df_long = df_long.dropna(subset=['GHG Emissions Per Capita'])

#Rename columns
df_long = df_long.rename(columns={
    df_long.columns[0]: 'CountryCode',
    df_long.columns[1]: 'Country'
})

print(df_long.head())


['CountryCode', 'Country', 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
  CountryCode      Country  Year  GHG Emissions Per Capita
0         ABW        Aruba  1970                  0.625934
1         AFG  Afghanistan  1970                  1.375142
2         AGO       Angola  1970                  2.779412
3         AIA     Anguilla  1970                  0.530614
4         ALB      Albania  1970                  3.827258


In [6]:
# countries list
countries = [
    'Australia', 'Austria', 'Brazil', 'Canada',
    'China', 'France', 'Germany',
    'Greece', 'Italy', 'Japan', 'South Korea',
    'Mexico', 'Norway', 'Russia', 'Spain',
    'United Kingdom', 'United States'
]

# Filter data
filtered_df = df_long[df_long['Country'].isin(countries)]

# Olympics dates
olympics = {
    'Australia': [1956, 2000],
    'Austria': [1964, 1976],
    'Brazil': [2016],
    'Canada': [1976, 1988, 2010],
    'China': [2008, 2022],
    'France': [1900, 1924, 1968, 1992, 2024],
    'Germany': [1936, 1972],
    'Greece': [1896, 2004],
    'Italy': [1956, 1960, 2006, 2026],
    'Japan': [1964, 1972, 1998, 2020],
    'South Korea': [1988, 2018],
    'Mexico': [1968],
    'Norway': [1952, 1994],
    'Russia': [1980, 2014],
    'Spain': [1992],
    'United Kingdom': [1908, 1948, 2012],
    'United States': [1932, 1960, 1980, 1984, 1996, 2002]
}

#Olympic host years dataframe
olympic_df = pd.DataFrame([
    {'Country': country, 'Year': year}
    for country, years in olympics.items()
    for year in years
])

#years filter
olympic_with_data = olympic_df.merge(
    filtered_df[['Country', 'Year']].drop_duplicates(),
    on=['Country', 'Year'],
    how='inner'
)

#tooltip
olympic_with_data['Event'] = 'Olympic Games'

#text labels at the top of the chart
y_max = filtered_df['GHG Emissions Per Capita'].max()
y_range = filtered_df['GHG Emissions Per Capita'].max() - filtered_df['GHG Emissions Per Capita'].min()
y_position = y_max + (y_range * 0.05)

text_df = olympic_with_data.copy()
text_df['y_position'] = y_position
text_df['label'] = text_df['Year'].astype(str)

#main chart tooltips
line_chart = alt.Chart(filtered_df).mark_line(
    strokeWidth=2
).encode(
    x=alt.X('Year:Q', title='Year', axis=alt.Axis(format='d')),
    y=alt.Y('GHG Emissions Per Capita:Q',
            title='GHG Emissions Per Capita (t CO2e)'),
    color=alt.Color('Country:N',
                    scale=alt.Scale(scheme='tableau20'),
                    legend=alt.Legend(orient='right', columns=2)),
    tooltip=[
        alt.Tooltip('Country:N', title='Country'),
        alt.Tooltip('Year:Q', title='Year', format='d'),
        alt.Tooltip('GHG Emissions Per Capita:Q', title='Emissions (t CO2e)', format='.2f')
    ]
)

#points but may not be needed Tooltip
points = alt.Chart(filtered_df).mark_point(
    size=1,
    filled=True,
    opacity=0.7
).encode(
    x='Year:Q',
    y='GHG Emissions Per Capita:Q',
    color='Country:N',
    tooltip=[
        alt.Tooltip('Country:N', title='Country'),
        alt.Tooltip('Year:Q', title='Year', format='d'),
        alt.Tooltip('GHG Emissions Per Capita:Q', title='Emissions (t CO2e)', format='.2f')
    ]
)

# Vertical lines
rules = alt.Chart(olympic_with_data).mark_rule(
    strokeDash=[4, 4],
    strokeWidth=1,
    opacity=0.3
).encode(
    x='Year:Q',
    color=alt.Color('Country:N',
                    scale=alt.Scale(scheme='tableau20'),
                    legend=None),
    tooltip=[
        alt.Tooltip('Country:N', title='Host Country'),
        alt.Tooltip('Year:Q', title='Olympic Year', format='d'),
        alt.Tooltip('Event:N', title='Event')
    ]
)

##labels Olympic years
text_labels = alt.Chart(text_df).mark_text(
    align='center',
    baseline='bottom',
    fontSize=8,
    fontWeight='bold',
    angle=0,
    dy=-5
).encode(
    x='Year:Q',
    y='y_position:Q',
    text='label:N',
    color=alt.Color('Country:N',
                    scale=alt.Scale(scheme='tableau20'),
                    legend=None)
)

# Combine all layers
final_chart = (line_chart + points + rules + text_labels).properties(
    width='container',
    height= 300,
    title='Per Capita GHG Emissions with Olympic Host Years'
).configure_axis(
    grid=True,
    gridOpacity=0.3
).configure_legend(
    title=None,
    labelFontSize=8
).configure_view(
    strokeOpacity=0
)

# Display the chart
final_chart.show()

alt.LayerChart(...)

In [ ]:
# # 3. Save the chart as HTML
final_chart.save('Emissions_Viz1.html')

In [8]:
# Allow more rows
alt.data_transformers.disable_max_rows()

# Your data processing
df_CO2.rename(columns={'EDGAR Country Code': 'CountryCode'}, inplace=True)

print(df_CO2.columns.tolist())

# Reshape from wide to long
year_columns = [col for col in df_CO2.columns if col not in ['CountryCode', 'Country']]

# Convert year columns to numeric
for col in year_columns:
    df_CO2[col] = pd.to_numeric(df_CO2[col], errors='coerce')

# Melt using column positions
df_long = df_CO2.melt(id_vars=[df_CO2.columns[0], df_CO2.columns[1]],
                       var_name='Year',
                       value_name='GHG Emissions Per Capita')

# Convert Year to numeric
df_long['Year'] = pd.to_numeric(df_long['Year'])

# Remove rows with NaN emissions
df_long = df_long.dropna(subset=['GHG Emissions Per Capita'])

# Rename the first two columns
df_long = df_long.rename(columns={
    df_long.columns[0]: 'CountryCode',
    df_long.columns[1]: 'Country'
})

print(f"Total rows: {len(df_long)}")
print(df_long.head())



# Countries list
countries = [
    'Australia', 'Austria', 'Brazil', 'Canada',
    'China', 'France', 'Germany',
    'Greece', 'Italy', 'Japan', 'South Korea',
    'Mexico', 'Norway', 'Russia', 'Spain',
    'United Kingdom', 'United States'
]

# Filter data
filtered_df = df_long[df_long['Country'].isin(countries)]

# print(f"Filtered rows: {len(filtered_df)}")
# print(f"Countries included: {filtered_df['Country'].nunique()}")


# Olympic host years

olympics = {
    'Australia': [1956, 2000],
    'Austria': [1964, 1976],
    'Brazil': [2016],
    'Canada': [1976, 1988, 2010],
    'China': [2008, 2022],
    'France': [1900, 1924, 1968, 1992, 2024],
    'Germany': [1936, 1972],
    'Greece': [1896, 2004],
    'Italy': [1956, 1960, 2006, 2026],
    'Japan': [1964, 1972, 1998, 2020],
    'South Korea': [1988, 2018],
    'Mexico': [1968],
    'Norway': [1952, 1994],
    'Russia': [1980, 2014],
    'Spain': [1992],
    'United Kingdom': [1908, 1948, 2012],
    'United States': [1932, 1960, 1980, 1984, 1996, 2002]
}

#  host years dataframe
olympic_df = pd.DataFrame([
    {'Country': country, 'Year': year}
    for country, years in olympics.items()
    for year in years
])

# filter Olympic years
olympic_with_data = olympic_df.merge(
    filtered_df[['Country', 'Year']].drop_duplicates(),
    on=['Country', 'Year'],
    how='inner'
)

# for tooltip
olympic_with_data['Event'] = 'Olympic Games'

# text labels
y_max = filtered_df['GHG Emissions Per Capita'].max()
y_range = filtered_df['GHG Emissions Per Capita'].max() - filtered_df['GHG Emissions Per Capita'].min()
y_position = y_max + (y_range * 0.05)

text_df = olympic_with_data.copy()
text_df['y_position'] = y_position
text_df['label'] = text_df['Year'].astype(str)


# trying to get multi-select to wrok
country_selection = alt.selection_point(
    fields=['Country'],
    bind='legend',
    name='country_selector',
    empty=True
)
## main chart

line_chart = alt.Chart(filtered_df).mark_line(
    strokeWidth=2
).encode(
    x=alt.X('Year:Q', title='Year', axis=alt.Axis(format='d')),
    y=alt.Y('GHG Emissions Per Capita:Q',
            title='Emissions Per Capita '),
    color=alt.Color('Country:N',
                    scale=alt.Scale(scheme='tableau20'),
                    legend=alt.Legend(
                        title="Select a Country",
                        orient='right',
                        columns=2,
                        labelFontSize=10,
                        titleFontSize=12
                    )),
    opacity=alt.condition(country_selection, alt.value(1), alt.value(0.15)),
    strokeWidth=alt.condition(country_selection, alt.value(3), alt.value(1)),
    tooltip=[
        alt.Tooltip('Country:N', title='Country'),
        alt.Tooltip('Year:Q', title='Year', format='d'),
        alt.Tooltip('GHG Emissions Per Capita:Q', title='Emissions', format='.2f')
    ]
).add_params(
    country_selection
)

# Marking points but may not be needed

points = alt.Chart(filtered_df).mark_point(
    size=1,
    filled=True,
    opacity=0.7
).encode(
    x='Year:Q',
    y='GHG Emissions Per Capita:Q',
    color='Country:N',
    opacity=alt.condition(country_selection, alt.value(0.9), alt.value(0.15)),
    tooltip=[
        alt.Tooltip('Country:N', title='Country'),
        alt.Tooltip('Year:Q', title='Year', format='d'),
        alt.Tooltip('GHG Emissions Per Capita:Q', title='Emissions', format='.2f')
    ]
)

##tooltip and rules

rules = alt.Chart(olympic_with_data).mark_rule(
    strokeDash=[4, 4],
    strokeWidth=1
).encode(
    x='Year:Q',
    color=alt.Color('Country:N',
                    scale=alt.Scale(scheme='tableau20'),
                    legend=None),
    opacity=alt.condition(country_selection, alt.value(0.7), alt.value(0.1)),
    tooltip=[
        alt.Tooltip('Country:N', title='Host Country'),
        alt.Tooltip('Year:Q', title='Olympic Year', format='d'),
        alt.Tooltip('Event:N', title='Event')
    ]
)
###Labeling

text_labels = alt.Chart(text_df).mark_text(
    align='center',
    baseline='bottom',
    fontSize=8,
    fontWeight='bold',
    angle=0,
    dy=-5
).encode(
    x='Year:Q',
    y='y_position:Q',
    text='label:N',
    color=alt.Color('Country:N',
                    scale=alt.Scale(scheme='tableau20'),
                    legend=None),
    opacity=alt.condition(country_selection, alt.value(0.9), alt.value(0.1))
)
# COMBINE ALL LAYERS

final_chart = (line_chart + points + rules + text_labels).properties(
    width='container',
    height=300,
    title='Per Capita Emissions'
).configure_axis(
    grid=True,
    gridOpacity=0.3
).configure_legend(
    titleFontSize=12,
    labelFontSize=10,
    orient='right',
    columns=2
).configure_view(
    strokeOpacity=0
).configure_title(
    fontSize=16,
    anchor='middle'
)


# Display the chart
final_chart.show()

# Save the chart
final_chart.save('Emissions_Viz1.1.html')


['CountryCode', 'Country', 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Total rows: 11495
  CountryCode      Country  Year  GHG Emissions Per Capita
0         ABW        Aruba  1970                  0.625934
1         AFG  Afghanistan  1970                  1.375142
2         AGO       Angola  1970                  2.779412
3         AIA     Anguilla  1970                  0.530614
4         ALB      Albania  1970                  3.827258


alt.LayerChart(...)

# Happiness

Load Data

In [9]:
# WHR26_Data_Figure_2.1

happiness_df= pd.read_excel("WHR26_Data_Figure_2.1.xlsx")
print(happiness_df.head())
happiness_df_2 = happiness_df[['Year','Rank', 'Country name']]
print(happiness_df_2.head())
print(happiness_df_2['Year'].unique())
unique_countries = sorted(happiness_df_2['Country name'].unique())
print(unique_countries)
print(happiness_df.shape)
happiness_df_2.shape

   Year  Rank Country name  Life evaluation (3-year average)  Lower whisker  \
0  2025     1      Finland                             7.764          7.690   
1  2025     2      Iceland                             7.540          7.449   
2  2025     3      Denmark                             7.539          7.446   
3  2025     4   Costa Rica                             7.439          7.356   
4  2025     5       Sweden                             7.255          7.172   

   Upper whisker  Explained by: Log GDP per capita  \
0          7.837                             1.915   
1          7.630                             1.971   
2          7.631                             1.986   
3          7.522                             1.697   
4          7.337                             1.950   

   Explained by: Social support  Explained by: Healthy life expectancy  \
0                         1.638                                  0.939   
1                         1.720                     

(2116, 3)

In [ ]:
happiness_df.columns

Index(['Year', 'Rank', 'Country name', 'Life evaluation (3-year average)',
       'Lower whisker', 'Upper whisker', 'Explained by: Log GDP per capita',
       'Explained by: Social support', 'Explained by: Healthy life expectancy',
       'Explained by: Freedom to make life choices',
       'Explained by: Generosity', 'Explained by: Perceptions of corruption',
       'Dystopia + residual'],
      dtype='object')

Plot

In [10]:
countries = [
    'Australia', 'Austria', 'Brazil', 'Canada',
    'China', 'France', 'Germany',
    'Greece', 'Italy', 'Japan', 'South Korea',
    'Mexico', 'Norway', 'Russia', 'Spain',
    'United Kingdom', 'United States'
]
# Olympics dates
olympics = {
    'Australia': [1956, 2000],
    'Austria': [1964, 1976],
    'Brazil': [2016],
    'Canada': [1976, 1988, 2010],
    'China': [2008, 2022],
    'France': [1900, 1924, 1968, 1992, 2024],
    'Germany': [1936, 1972],
    'Greece': [1896, 2004],
    'Italy': [1956, 1960, 2006, 2026],
    'Japan': [1964, 1972, 1998, 2020],
    'South Korea': [1988, 2018],
    'Mexico': [1968],
    'Norway': [1952, 1994],
    'Russia': [1980, 2014],
    'Spain': [1992],
    'United Kingdom': [1908, 1948, 2012],
    'United States': [1932, 1960, 1980, 1984, 1996, 2002]
}

In [11]:
selected_countries = [
    'Australia', 'Austria', 'Brazil', 'Canada',
    'China', 'France', 'Germany',
    'Greece', 'Italy', 'Japan', 'South Korea',
    'Mexico', 'Norway', 'Russia', 'Spain',
    'United Kingdom', 'United States'
]

# Filter data
filtered_df = happiness_df[happiness_df['Country name'].isin(selected_countries)]

# Reshape data from wide to long format
long_df = pd.melt(
    filtered_df,
    id_vars=['Year', 'Country name'],
    value_vars=[
        'Explained by: Social support',
        'Explained by: Healthy life expectancy'
    ],
    var_name='Metric',
    value_name='Score'
)

long_df['Year'] = pd.to_datetime(long_df['Year'], format='%Y')

#faceted chart
chart = alt.Chart(long_df).mark_line(point=True).encode(
    x=alt.X('Year:T', title='Year', axis=alt.Axis(format='%Y')),
    y=alt.Y('Score:Q', title='Score', scale=alt.Scale(zero=False)),
    color=alt.Color('Country name:N',
                    legend=alt.Legend(title='Country'),
                    scale=alt.Scale(scheme='category10')),
    facet=alt.Facet('Metric:N', columns=2,
                    title='Metric Comparison'),
    tooltip=['Country name', alt.Tooltip('Year:T', format='%Y'), 'Metric', 'Score']
).properties(
    title='Social Support vs Life Expectancy Over Time',
    width=400,
    height=400
)

chart.show()

alt.Chart(...)

In [12]:
alt.data_transformers.disable_max_rows()

selected_countries = [
    'Australia', 'Austria', 'Brazil', 'Canada',
    'China', 'France', 'Germany',
    'Greece', 'Italy', 'Japan', 'South Korea',
    'Mexico', 'Norway', 'Russia', 'Spain',
    'United Kingdom', 'United States'
]
# Create dropdown binding
country_dropdown = alt.binding_select(
    options=selected_countries,
    name='Select Country: '
)

# Create selection
country_selection = alt.selection_point(
    fields=['Country name'],
    bind=country_dropdown,
    value=[{'Country name': selected_countries[0]}]
)

#5 metrics
metrics = [
    'Explained by: Social support',
    'Explained by: Healthy life expectancy',
    'Explained by: Freedom to make life choices',
    'Explained by: Generosity',
    'Explained by: Perceptions of corruption'
]

#reshape data
long_df = pd.melt(
    happiness_df,
    id_vars=['Year', 'Country name'],
    value_vars=metrics,
    var_name='Metric',
    value_name='Score'
)

#names for display
long_df['Metric'] = long_df['Metric'].str.replace('Explained by: ', '')

long_df['Year'] = pd.to_datetime(long_df['Year'], format='%Y')

#faceted chart
chart = alt.Chart(long_df).mark_line(
    point=alt.OverlayMarkDef(
        filled=True,
        fill='white',
        size=40
    ),
    strokeWidth=2.5
).encode(
    x=alt.X('Year:T',
            title='Year',
            axis=alt.Axis(format='%Y', labelAngle=0, grid=True),
            scale=alt.Scale(
                domain=[pd.to_datetime('2019'), pd.to_datetime('2025')]
    )),
    y=alt.Y('Score:Q',
            title='Score',
            scale=alt.Scale(zero=False),
            axis=alt.Axis(grid=True)),
    facet=alt.Facet('Metric:N',
                    columns=3,
                    title='',
                    header=alt.Header(
                        labelFontSize=13,
                        labelFontWeight='bold',
                        labelColor='#333'
                    )),
    tooltip=[
        alt.Tooltip('Year:T', format='%Y', title='Year'),
        alt.Tooltip('Score:Q', format='.3f', title='Score'),
        alt.Tooltip('Metric:N', title='Metric')
    ]
).add_params(
    country_selection
).transform_filter(
    country_selection
).properties(
    title=alt.TitleParams(
        text='Happiness Metrics Over Time',
        subtitle='Select a country to see all 5 metrics side by side',
        fontSize=18,
        subtitleFontSize=13,
        anchor='middle'
    ),
    width=250,
    height=250,
    background='white'
).configure_view(
    stroke='lightgray',
    strokeWidth=0.5
).configure_axis(
    labelFontSize=11,
    titleFontSize=12
)

chart.show()

alt.Chart(...)

In [13]:
## iterating again!!!
# Allow more rows
alt.data_transformers.disable_max_rows()

#data validation for countries because it was breaking
available_countries = [c for c in countries if c in happiness_df_2['Country name'].unique()]
print("Available countries:", available_countries)

#filter
filtered_df = happiness_df_2[happiness_df_2['Country name'].isin(available_countries)]

#filter year
filtered_df = filtered_df[filtered_df['Year'] >= 2012]

# print(f"Filtered rows: {len(filtered_df)}")


#sort
filtered_df = filtered_df.sort_values(['Country name', 'Year'])

###percentage change in
filtered_df['Rank_Change'] = filtered_df.groupby('Country name')['Rank'].pct_change() * 100
filtered_df['Rank_Change_Label'] = filtered_df['Rank_Change'].apply(
    lambda x: f"{x:.1f}%" if pd.notnull(x) else "N/A"
)

#create label
filtered_df['Tooltip_Label'] = filtered_df.apply(
    lambda row: f"{row['Country name']}\nRank: {row['Rank']}\nChange: {row['Rank_Change_Label']}",
    axis=1
)

##only years >= 2012 because the metrics arent shared for previous years
olympic_data = []
for country, years in olympics.items():
    for year in years:
        if year >= 2012:
            olympic_data.append({
                'Country': country,
                'Year': year
            })

olympic_df = pd.DataFrame(olympic_data)

#filter and validation since it wasnt working right
if len(olympic_df) > 0:
    olympic_with_data = olympic_df.merge(
        filtered_df[['Country name', 'Year']].drop_duplicates(),
        left_on=['Country', 'Year'],
        right_on=['Country name', 'Year'],
        how='inner'
    )
    print(f"Olympic markers: {len(olympic_with_data)}")
else:
    olympic_with_data = pd.DataFrame()


#trying to get multi-select to work
country_selection = alt.selection_point(
    fields=['Country name'],
    bind='legend',
    name='country_selector'
)
#chart filter and tooltip
line_chart = alt.Chart(filtered_df).mark_line(
    strokeWidth=2
).encode(
    x=alt.X('Year:Q', title='Year', axis=alt.Axis(format='d')),
    y=alt.Y('Rank:Q',
            title='Happiness Rank',
            scale=alt.Scale(reverse=True)),
    color=alt.Color('Country name:N',
                    scale=alt.Scale(scheme='tableau20'),
                    legend=alt.Legend(
                        orient='right',
                        columns=2,
                        labelFontSize=8,
                        title="Select a Country"
                    )),
    opacity=alt.condition(country_selection, alt.value(1), alt.value(0.1)),
    strokeWidth=alt.condition(country_selection, alt.value(2.5), alt.value(1)),
    tooltip=[
        alt.Tooltip('Country name:N', title='Country'),
        alt.Tooltip('Year:Q', title='Year'),
        alt.Tooltip('Rank:Q', title='Rank'),
        alt.Tooltip('Rank_Change_Label:O', title='% Change from Last Year')
    ]
).add_params(
    country_selection
)

# Points with tooltip
points = alt.Chart(filtered_df).mark_point(
    size=1,
    opacity=0.7
).encode(
    x='Year:Q',
    y='Rank:Q',
    color='Country name:N',
    opacity=alt.condition(country_selection, alt.value(0.9), alt.value(0.1)),
    tooltip=[
        alt.Tooltip('Country name:N', title='Country'),
        alt.Tooltip('Year:Q', title='Year'),
        alt.Tooltip('Rank:Q', title='Rank'),
        alt.Tooltip('Rank_Change_Label:O', title='% Change from Last Year')
    ]
)

#base chart
final_chart = (line_chart + points).properties(
    width=800,
    height=500,
    title='Happiness Rank Over Time'
).configure_axis(
    grid=True,
    gridOpacity=0.3
).configure_legend(
    title=None,
    labelFontSize=8,
    orient='right',
    columns=2
).configure_view(
    strokeOpacity=0
)

if len(olympic_with_data) > 0:
    #y-axis range for text placement
    y_min = filtered_df['Rank'].min()
    y_max = filtered_df['Rank'].max()
    y_range = y_max - y_min
    y_position = y_max * 0.95

    text_df = olympic_with_data.copy()
    text_df['y_position'] = y_position
    text_df['label'] = text_df['Country'] + '\n' + text_df['Year'].astype(str)

    #vertical  for Olympic years
    rules = alt.Chart(olympic_with_data).mark_rule(
        strokeDash=[4, 4],
        strokeWidth=1.5,
        opacity=0.6
    ).encode(
        x='Year:Q',
        color=alt.Color('Country:N',scale=alt.Scale(scheme='tableau20'),legend=None))

    ##label years
    text_labels = alt.Chart(text_df).mark_text(
        align='center',
        baseline='bottom',
        fontSize=9,
        fontWeight='bold',
        lineBreak='\n',
        dy=-5
    ).encode(
        x='Year:Q',
        y='y_position:Q',
        text='label:N',
        color=alt.Color('Country:N',
                        scale=alt.Scale(scheme='tableau20'),
                        legend=None)
    )

    ### CombineOlympic markers
    final_chart = (line_chart + points + rules + text_labels).properties(
        width='container',
        height=300,
        title='Happiness Rank Over Time'
    ).configure_axis(
        grid=True,
        gridOpacity=0.3
    ).configure_legend(
        title=None,
        labelFontSize=8,
        orient='right',
        columns=2
    ).configure_view(
        strokeOpacity=0
    )


# Display the chart
final_chart.show()

# Save the chart
final_chart.save('Happiness_Viz3.html')


Available countries: ['Australia', 'Austria', 'Brazil', 'Canada', 'China', 'France', 'Germany', 'Greece', 'Italy', 'Japan', 'Mexico', 'Norway', 'Spain', 'United Kingdom', 'United States']
Olympic markers: 5


alt.LayerChart(...)

Extreme Testing space

'Year', 'Rank',
- 'Country name',
- 'Explained by: Log GDP per capita',
- 'Explained by: Social support',
- 'Explained by: Healthy life expectancy',
- 'Explained by: Freedom to make life choices',
- 'Explained by: Generosity',
- 'Explained by: Perceptions of corruption'

In [14]:
original_df = happiness_df.copy()
selected_countries = [
    'Australia', 'Austria', 'Brazil', 'Canada',
    'China', 'France', 'Germany',
    'Greece', 'Italy', 'Japan', 'South Korea',
    'Mexico', 'Norway', 'Russia', 'Spain',
    'United Kingdom', 'United States'
]
selected_years = list(range(2018, 2025))
# Filter data
happiness_df_final = happiness_df[happiness_df['Country name'].isin(selected_countries)]
happiness_df_final = happiness_df_final[happiness_df_final['Year'].isin(selected_years)]

In [15]:
# Rename and keep all columns
happiness_df_final = happiness_df_final.rename(columns={'Country name': 'Country',
    'Explained by: Log GDP per capita': 'GDP per capita',
    'Explained by: Social support': 'Social support',
    'Explained by: Healthy life expectancy': 'Healthy life expectancy',
    'Explained by: Freedom to make life choices': 'Freedom to make life choices',
    'Explained by: Generosity': 'Generosity',
    'Explained by: Perceptions of corruption': 'Perceptions of corruption'
})

In [16]:

olympics_since_2018 = {
    'China': [2022],
    'France': [2024],
    'Italy': [2026],
    'Japan': [2020],
    'South Korea': [2018]
}

countries = [ 'Australia', 'Austria', 'Brazil', 'Canada',
    'China', 'France', 'Germany',
    'Greece', 'Italy', 'Japan', 'South Korea',
    'Mexico', 'Norway', 'Russia', 'Spain',
    'United Kingdom', 'United States']
years = list(range(2018, 2025))
# metrics = ['GDP per capita', 'Social support', 'Healthy life expectancy', 'Freedom to make life choices', 'Generosity', 'Perceptions of corruption']

df = happiness_df_final

# Create a dropdown selector for country
country_select = alt.selection_point(
    fields=['Country'],
    bind=alt.binding_select(options=countries, name='Select Country: '),
    value=[{'Country': 'United States'}]
)

# Define the metrics to plot
metrics_list = ['GDP per capita', 'Social support', 'Healthy life expectancy', 'Freedom to make life choices', 'Generosity', 'Perceptions of corruption']

olympic_data = []
for country, years_list in olympics_since_2018.items():
    for year in years_list:
        olympic_data.append({'Country': country, 'Olympic_Year': year})
olympic_df = pd.DataFrame(olympic_data)


# Create individual charts
charts = []
for metric in metrics_list:
    # Determine y-axis title and scale
    y_title = metric
    if metric == 'GDP per capita':
        y_title = 'GDP per capita'
    elif metric == 'Social support':
        y_title = 'Social support'
    elif metric == 'Healthy life expectancy':
        y_title = 'Healthy life expectancy'
    elif metric == 'Freedom to make life choices':
        y_title = 'Freedom to make life choices'
    elif metric == 'Generosity':
        y_title = 'Generosity'
    elif metric == 'Perceptions of corruption':
        y_title = 'Perceptions of corruption'

    chart = alt.Chart(df).mark_line(point=True, strokeWidth=2).encode(
        x=alt.X('Year:Q',
                title='Year',
                scale=alt.Scale(zero=False),
                axis=alt.Axis(format='d')),
        y=alt.Y(f'{metric}:Q', title=y_title, scale=alt.Scale(zero=False)),
        color=alt.condition(
            country_select,
            alt.Color('Country:N', legend=None),
            alt.value('lightgray')
        ),
        opacity=alt.condition(
            country_select,
            alt.value(1.0),
            alt.value(0.2)
        ),
        tooltip=['Country', 'Year', metric]
    ).add_params(
        country_select
    ).properties(
        width=200,
        height=150,
        title=metric
    )

    charts.append(chart)

# 2x3 grid
grid = alt.hconcat(
    alt.vconcat(charts[0], charts[1]),
    alt.vconcat(charts[2], charts[3]),
    alt.vconcat(charts[4], charts[5])
).properties(
    title={
        'text': 'Country Metrics Over Time (2015-2024)',
        'fontSize': 20,
        'anchor': 'middle'
    }
)

# Display the grid
grid

alt.HConcatChart(...)

In [17]:
#filter for just these years
olympics_since_2018 = {
    'China': {'city': 'Beijing', 'year': 2022},
    'France': {'city': 'Paris', 'year': 2024},
    'Japan': {'city': 'Tokyo', 'year': 2020},
}

countries = ['Australia', 'Austria', 'Brazil', 'Canada',
    'China', 'France', 'Germany',
    'Greece', 'Italy', 'Japan',
    'Mexico', 'Norway',  'Spain',
    'United Kingdom', 'United States']

df = happiness_df_final

#selector
country_select = alt.selection_point(
    fields=['Country'],
    bind=alt.binding_select(options=countries, name='Select Country: '),
    value=[{'Country': 'United States'}]
)

#metrics to plot
metrics_list = ['GDP per capita', 'Social support', 'Healthy life expectancy',
                'Freedom to make life choices', 'Generosity', 'Perceptions of corruption']

###data frame with olympic years and host cities
olympic_data = []
for country, info in olympics_since_2018.items():
    olympic_data.append({
        'Country': country,
        'Olympic_Year': info['year'],
        'Host_City': info['city']
    })
olympic_df = pd.DataFrame(olympic_data)

#individual chart creation
charts = []
for metric in metrics_list:
    #y-axis title
    y_title = f"{metric} score"

    #main chart
    base_chart = alt.Chart(df).mark_line(point=True, strokeWidth=2).encode(
        x=alt.X('Year:Q',
                title='Year',
                scale=alt.Scale(zero=False, domain=[2018, 2025]),
                axis=alt.Axis(format='d', values=list(range(2018, 2025, 2)))),
        y=alt.Y(f'{metric}:Q', title=y_title, scale=alt.Scale(zero=False)),
        color=alt.condition(
            country_select,
            alt.Color('Country:N', legend=None),
            alt.value('steelblue')
        ),
        opacity=alt.condition(
            country_select,
            alt.value(1.0),
            alt.value(0.1)
        ),
        tooltip=['Country', 'Year', metric]
    ).add_params(
        country_select
    )

    #vertical lines for Olympic years
    olympic_lines = alt.Chart(olympic_df).mark_rule(
        color='blue',
        strokeDash=[4, 4],
        strokeWidth=1.5,
        opacity=0.7,
        size=1
    ).encode(
        x='Olympic_Year:Q',
        tooltip=['Country', 'Olympic_Year', 'Host_City']
    )

    #labels
    olympic_labels = alt.Chart(olympic_df).mark_text(
        color='blue',
        fontSize=8,
        opacity=0.7,
        angle=45,  # Angled labels to be more legiable
        align='left',
        baseline='top',
        dx=5,
        dy=0
    ).encode(
        x='Olympic_Year:Q',
        y=alt.value(15),
        text='Host_City:N'
    )

    #layer
    chart = (base_chart + olympic_lines + olympic_labels).properties(
        width=200,
        height=150,
        title=metric
    )

    charts.append(chart)

#3x2 grid
grid = alt.vconcat(
    alt.hconcat(charts[0], charts[1], charts[2]),
    alt.hconcat(charts[3], charts[4], charts[5])
).properties(
    title={
        'text': 'Country Metrics Over Time (2018-2025)',
        'fontSize': 20,
        'anchor': 'middle'
    }
)

# Display
grid

alt.VConcatChart(...)

In [ ]:
### hopefully last iteration ###
# Olympic host cities with their years and countries
olympics_since_2018 = {
    'China': {'city': 'Beijing', 'year': 2022},
    'France': {'city': 'Paris', 'year': 2024},
    'Japan': {'city': 'Tokyo', 'year': 2020},
}

countries = ['Australia', 'Austria', 'Brazil', 'Canada',
    'China', 'France', 'Germany',
    'Greece', 'Italy', 'Japan', 'South Korea',
    'Mexico', 'Norway', 'Russia', 'Spain',
    'United Kingdom', 'United States']

df = happiness_df_final

# Create a dropdown selector for country
country_select = alt.selection_point(
    fields=['Country'],
    bind=alt.binding_select(options=countries, name='Select Country: '),
    value=[{'Country': 'United States'}]
)

# Define the metrics to plot
metrics_list = ['GDP per capita', 'Social support', 'Healthy life expectancy',
                'Freedom to make life choices', 'Generosity', 'Perceptions of corruption']

olympic_data = []
for country, info in olympics_since_2018.items():
    olympic_data.append({
        'Country': country,
        'Olympic_Year': info['year'],
        'Host_City': info['city']
    })
olympic_df = pd.DataFrame(olympic_data)

rank_chart = alt.Chart(df).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X('Year:Q',
            title='Year',
            scale=alt.Scale(zero=False, domain=[2018, 2025]),
            axis=alt.Axis(format='d', values=list(range(2018, 2025, 2)))),
    y=alt.Y('Rank:Q',
            title='Rank (lower is better)',
            scale=alt.Scale(zero=False, reverse=True)),  # one at the top
    color=alt.condition(
        country_select,
        alt.Color('Country:N', legend=None),
        alt.value('steelblue')
    ),
    opacity=alt.condition(
        country_select,
        alt.value(1.0),
        alt.value(0.1)
    ),
    tooltip=['Country', 'Year', 'Rank']
).add_params(
    country_select
)

#lines for rank chart
rank_olympic_lines = alt.Chart(olympic_df).mark_rule(
    color='blue',
    strokeDash=[4, 4],
    strokeWidth=1.5,
    opacity=0.7
).encode(
    x='Olympic_Year:Q',
    tooltip=['Country', 'Olympic_Year', 'Host_City']
)

#labels
rank_olympic_labels = alt.Chart(olympic_df).mark_text(
    color='blue',
    fontSize=10,
    opacity=0.7,
    angle=45,
    align='left',
    baseline='top',
    dx=5,
    dy=0
).encode(
    x='Olympic_Year:Q',
    y=alt.value(15),
    text='Host_City:N'
)

#combined
rank_chart_final = (rank_chart + rank_olympic_lines + rank_olympic_labels).properties(
    width=300,  # Wider than individual metric charts
    height=500,  # Taller to match the full grid height
    title='Happiness Rank Over Time'
)


##individual metric charts
metric_charts = []
for metric in metrics_list:
    # y-axis title
    y_title = f"{metric} score"

    #Base chart
    base_chart = alt.Chart(df).mark_line(point=True, strokeWidth=2).encode(
        x=alt.X('Year:Q',
                title='Year',
                scale=alt.Scale(zero=False, domain=[2018, 2025]),
                axis=alt.Axis(format='d', values=list(range(2018, 2025, 2)))),
        y=alt.Y(f'{metric}:Q', title=y_title, scale=alt.Scale(zero=False)),
        color=alt.condition(
            country_select,
            alt.Color('Country:N', legend=None),
            alt.value('steelblue')
        ),
        opacity=alt.condition(
            country_select,
            alt.value(1.0),
            alt.value(0.1)
        ),
        tooltip=['Country', 'Year', metric]
    ).add_params(
        country_select
    )

    ###lines for Olympic years
    olympic_lines = alt.Chart(olympic_df).mark_rule(
        color='blue',
        strokeDash=[4, 4],
        strokeWidth=1.5,
        opacity=0.7,
        size=1
    ).encode(
        x='Olympic_Year:Q',
        tooltip=['Country', 'Olympic_Year', 'Host_City']
    )

    #Add labels
    olympic_labels = alt.Chart(olympic_df).mark_text(
        color='blue',
        fontSize=8,
        opacity=0.7,
        angle=45,
        align='left',
        baseline='top',
        dx=5,
        dy=0
    ).encode(
        x='Olympic_Year:Q',
        y=alt.value(15),
        text='Host_City:N'
    )

    #Layer the charts
    chart = (base_chart + olympic_lines + olympic_labels).properties(
        width=200,
        height=150,
        title=metric
    )

    metric_charts.append(chart)

#3x2 grid
metric_grid = alt.vconcat(
    alt.hconcat(metric_charts[0], metric_charts[1], metric_charts[2]),
    alt.hconcat(metric_charts[3], metric_charts[4], metric_charts[5])
)

#plot
final_grid = alt.hconcat(
    rank_chart_final,
    metric_grid
).properties(
    title={
        'text': 'Country Metrics Over Time (2018-2025)',
        'fontSize': 20,
        'anchor': 'middle'
    }
)

# Display the grid
final_grid

#Save the chart
# final_grid.save('Happiness_Viz4=.html')



alt.HConcatChart(...)